[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wellingtonmnf/pos-graduacao-ia-ciencia-dados/blob/main/disciplinas/engenharia-preparacao-visualizacao-dados/notebooks/k-means-random-state.ipynb)


In [2]:
# Revisão documental — 20/10/2026.
# Saídas históricas preservadas. As saídas de C4/C5/C9/C10/C11/C12 precedem as correções dos rótulos,
# do alinhamento dos centros e da comparação por ARI; use a explicação da fonte atual. Execução
# integral desta revisão no Colab pendente.

import numpy as np
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
import matplotlib.pyplot as plt

In [3]:
# Vamos criar um dataset simples para demonstração
X, _ = make_blobs(n_samples=100, centers=4, cluster_std=1.0,
                  center_box=(0.0, 10.0), random_state=42)

print("=== DEMONSTRAÇÃO DO PARÂMETRO random_state=0 NO KMEANS ===\n")

=== DEMONSTRAÇÃO DO PARÂMETRO random_state=0 NO KMEANS ===



In [4]:
# 1. Ilustração de uma sequência pseudoaleatória NumPy
print("1. ILUSTRAÇÃO DO GERADOR NUMPY")
print("-" * 50)

1. INVESTIGANDO A SEQUÊNCIA DE NÚMEROS ALEATÓRIOS
--------------------------------------------------


In [5]:
# Ilustração independente: não é um rastreamento dos números consumidos pelo KMeans.
np.random.seed(0)
print("Primeiros 10 números do gerador NumPy com seed=0:")
print(np.random.random(10))

print("\n2. COMPARANDO EXECUÇÕES COM E SEM random_state")
print("-" * 50)

Primeiros 10 números aleatórios com seed=0:
[0.5488135  0.71518937 0.60276338 0.54488318 0.4236548  0.64589411
 0.43758721 0.891773   0.96366276 0.38344152]

2. COMPARANDO EXECUÇÕES COM E SEM random_state
--------------------------------------------------


In [6]:
# Execução 1: COM random_state=0
kmeans_1 = KMeans(n_clusters=3, random_state=0, n_init=10)
labels_1 = kmeans_1.fit_predict(X)
centers_1 = kmeans_1.cluster_centers_

In [7]:
# Execução 2: COM random_state=0 (deve ser idêntica)
kmeans_2 = KMeans(n_clusters=3, random_state=0, n_init=10)
labels_2 = kmeans_2.fit_predict(X)
centers_2 = kmeans_2.cluster_centers_

In [8]:
# Execução 3: SEM random_state (resultados podem variar)
kmeans_3 = KMeans(n_clusters=3, n_init=10)
labels_3 = kmeans_3.fit_predict(X)
centers_3 = kmeans_3.cluster_centers_

print("Centroides - Execução 1 (random_state=0):")
print(centers_1)
print("\nCentroides - Execução 2 (random_state=0):")
print(centers_2)
print("\nCentroides - Execução 3 (sem random_state):")
print(centers_3)

Centroides - Execução 1 (random_state=0):
[[2.06043899 9.06129871]
 [7.12447823 6.03463664]
 [1.46486669 1.67533513]]

Centroides - Execução 2 (random_state=0):
[[2.06043899 9.06129871]
 [7.12447823 6.03463664]
 [1.46486669 1.67533513]]

Centroides - Execução 3 (sem random_state):
[[1.46486669 1.67533513]
 [7.12447823 6.03463664]
 [2.06043899 9.06129871]]


In [9]:
# As saídas históricas foram preservadas; as comparações abaixo usam a execução atual.
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from sklearn.metrics import adjusted_rand_score

def mesmos_centros(centros_a, centros_b):
    """Compara os centros com correspondência entre clusters, independentemente da ordem."""
    if centros_a.shape != centros_b.shape:
        return False
    linhas, colunas = linear_sum_assignment(cdist(centros_a, centros_b))
    return bool(np.allclose(centros_a[linhas], centros_b[colunas]))

print(f"\nCentros 1 e 2 na mesma ordem? {np.allclose(centers_1, centers_2)}")
print(f"Centros 1 e 3 na mesma ordem? {np.allclose(centers_1, centers_3)}")
print(f"Mesmos centros entre 1 e 2, independentemente da ordem? {mesmos_centros(centers_1, centers_2)}")
print(f"Mesmos centros entre 1 e 3, independentemente da ordem? {mesmos_centros(centers_1, centers_3)}")
print(f"ARI entre as partições 1 e 2: {adjusted_rand_score(labels_1, labels_2):.6f}")
print(f"ARI entre as partições 1 e 3: {adjusted_rand_score(labels_1, labels_3):.6f}")
print("ARI = 1 indica a mesma partição destas amostras, mesmo que os clusters sejam renumerados.")
print("Uma comparação ordenada falsa não basta para concluir que os centros ou agrupamentos mudaram.")

print("\n3. CENTROIDES APÓS UMA ITERAÇÃO")
print("-" * 50)



Executions 1 e 2 são idênticas? True
Executions 1 e 3 são idênticas? False

3. ANALISANDO OS VALORES INTERNOS DO ALGORITMO
--------------------------------------------------


In [10]:
# Observamos os centros após uma atualização; isto não captura a inicialização.
kmeans_detailed = KMeans(n_clusters=3, random_state=0, n_init=1, max_iter=1)
kmeans_detailed.fit(X)

print("Centroides após uma iteração com random_state=0:")
print(kmeans_detailed.cluster_centers_)

print("\nInércia (soma das distâncias quadráticas aos centroides):")
print(f"{kmeans_detailed.inertia_:.4f}")

print("\n4. DEMONSTRAÇÃO DA REPRODUTIBILIDADE")
print("-" * 50)

Centroides iniciais (após 1 iteração com random_state=0):
[[1.80471791 9.16915179]
 [6.66405332 6.37730069]
 [1.46486669 1.67533513]]

Inércia (soma das distâncias quadráticas aos centroides):
293.6936

4. DEMONSTRAÇÃO DA REPRODUTIBILIDADE
--------------------------------------------------


In [11]:
# Comparando as inércias de ajustes com os mesmos dados, parâmetros e ambiente.
results = []
for i in range(5):
    km = KMeans(n_clusters=3, random_state=0, n_init=10)
    km.fit(X)
    results.append(km.inertia_)
    print(f"Execução {i+1} - Inércia: {km.inertia_:.4f}")

print(f"\nAs inércias observadas são iguais? {len(set(results)) == 1}")

print("\n5. SEQUÊNCIA NUMPY ILUSTRATIVA")
print("-" * 50)

Execução 1 - Inércia: 283.0549
Execução 2 - Inércia: 283.0549
Execução 3 - Inércia: 283.0549
Execução 4 - Inércia: 283.0549
Execução 5 - Inércia: 283.0549

Todas as execuções são idênticas? True

5. NÚMEROS ALEATÓRIOS ESPECÍFICOS USADOS INTERNAMENTE
--------------------------------------------------


In [12]:
# Esta sequência ilustra o determinismo do gerador NumPy; não instrumenta o KMeans.
np.random.seed(0)
print("Sequência ilustrativa NumPy com seed=0:")
random_numbers = np.random.random(20)
for i, num in enumerate(random_numbers):
    print(f"Posição {i:2d}: {num:.10f}")

print(f"\nPrimeiro número: {random_numbers[0]:.10f}")
print(f"Último número:   {random_numbers[-1]:.10f}")

print("\n" + "="*60)
print("CONCLUSÃO:")
print("- random_state inteiro fixa a aleatoriedade controlada pelo estimador.")
print("- A comparação de repetibilidade vale para os mesmos dados, parâmetros e ambiente.")
print("- A sequência NumPy acima é ilustrativa; não lista os números consumidos pelo KMeans.")
print("- Inércias iguais, isoladamente, não comprovam centros ou partições iguais.")
print("- Uma execução sem semente pode obter a mesma solução, apenas renumerá-la ou produzir outra solução.")
print("- Saídas históricas preservadas; estes novos textos não foram executados no notebook original.")
print("="*60)


Sequência de números com seed=0 (primeira execução):
Posição  0: 0.5488135039
Posição  1: 0.7151893664
Posição  2: 0.6027633761
Posição  3: 0.5448831830
Posição  4: 0.4236547993
Posição  5: 0.6458941131
Posição  6: 0.4375872113
Posição  7: 0.8917730008
Posição  8: 0.9636627605
Posição  9: 0.3834415188
Posição 10: 0.7917250381
Posição 11: 0.5288949198
Posição 12: 0.5680445611
Posição 13: 0.9255966383
Posição 14: 0.0710360582
Posição 15: 0.0871292997
Posição 16: 0.0202183974
Posição 17: 0.8326198455
Posição 18: 0.7781567509
Posição 19: 0.8700121482

Primeiro número: 0.5488135039
Último número:   0.8700121482

CONCLUSÃO:
- random_state=0 garante que os mesmos números aleatórios
  sejam gerados a cada execução
- Isso torna os resultados completamente reproduzíveis
- Fundamental para pesquisa e desenvolvimento científico!
